In [ ]:
import pandas as pd
import numpy as np
import random
from pandasql import sqldf
import duckdb
from jinja2 import Template
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [1452]:
df = pd.read_csv(r'C:\Users\Павел\Desktop\OTUS\marketplace_catalog.csv')
df.head(1)

,product_id,product_name,category_path,brand,seller_id,price,discount_pct,final_price,specifications,description,is_in_stock,rating,reviews_count,created_date
0,PRD105397,SAMSUNG Планшеты Max,Электроника -> Планшеты -> Аксессуары,SAMSUNG,SELL6224,141731,40,85038,"{""характеристика"": ""не указано""}",Качественный товар SAMSUNG для повседневного использования.,False,4.0,286,2025-06-25


In [1454]:
df.shape

(10300, 14)

In [1456]:
#  Для обращения к датафрейму с помощью SQL, используем duckdb.
con = duckdb.connect()
con.register('products', df)

In [1458]:
df.columns

Index(['product_id', 'product_name', 'category_path', 'brand', 'seller_id',
       'price', 'discount_pct', 'final_price', 'specifications', 'description',
       'is_in_stock', 'rating', 'reviews_count', 'created_date'],
      dtype='object')

In [1460]:
#1. Определяем количество незаполненных полей по каждому атрибуту
columns = df.columns.tolist() 
sql_template = """
    SELECT 'completeness' AS dq_dimension
         , metric AS entity
         , ROUND(value / total_rows, 3) AS metric_value
         , CONCAT('Non-nulls: ', value, ' Total rows: ', total_rows) AS details
      FROM 
          (
           SELECT COUNT(*) AS total_rows
                    {% for col in columns %}
                , COUNT({{ col }}) AS {{ col }}
                    {% endfor %}
             FROM products
          ) s
   UNPIVOT 
          (
            value FOR metric IN 
                               (
                                     {% for col in columns %}
                                 {{ col }}{% if not loop.last %},{% endif %}
                                     {% endfor %}
                               )
          ) AS u
  ORDER BY metric_value ASC;
"""
sql_template = Template(sql_template)
completeness_query = sql_template.render(columns=columns)
completeness = con.execute(completeness_query).fetchdf()
completeness

,dq_dimension,entity,metric_value,details
0,completeness,specifications,0.880,Non-nulls: 9059 Total rows: 10300
1,completeness,description,0.899,Non-nulls: 9261 Total rows: 10300
2,completeness,brand,0.919,Non-nulls: 9465 Total rows: 10300
3,completeness,product_id,1.000,Non-nulls: 10300 Total rows: 10300
4,completeness,product_name,1.000,Non-nulls: 10300 Total rows: 10300
5,completeness,category_path,1.000,Non-nulls: 10300 Total rows: 10300
6,completeness,seller_id,1.000,Non-nulls: 10300 Total rows: 10300
7,completeness,price,1.000,Non-nulls: 10300 Total rows: 10300
8,completeness,discount_pct,1.000,Non-nulls: 10300 Total rows: 10300
9,completeness,final_price,1.000,Non-nulls: 10300 Total rows: 10300


In [1462]:
#2. Определяем количество дубликатов по каждому атрибуту + их комбинациям - sql
# Определяем метрики как список словарей
metrics_config = [
    {
        'name': 'product_id',
        'distinct_expr': 'product_id',
        'group_by': ['product_id'],
        'example_expr': "'Дубликаты ID: ' || product_id::TEXT || ' (' || COUNT(*) || ' times)'"
    },
    {
        'name': 'product_name',
        'distinct_expr': 'product_name',
        'group_by': ['product_name'],
        'example_expr': "product_name || ' (' || COUNT(*) || ' раз)'"
    },
    {
        'name': 'product_name + brand',
        'distinct_expr': "product_name || '|' || COALESCE(brand, '')",
        'group_by': ['product_name', 'brand'],
        'example_expr': "product_name || ' | ' || COALESCE(brand, 'NULL') || ' (' || COUNT(*) || ' times)'"
    },
    {
        'name': 'category + price + brand',
        'distinct_expr': "category_path || '|' || price::TEXT || '|' || COALESCE(brand, '')",
        'group_by': ['category_path', 'price', 'brand'],
        'example_expr': "category_path || ' | ' || price::TEXT || ' | ' || COALESCE(brand, 'NULL') || ' (' || COUNT(*) || ' times)'"
    }
]

sql_template = """

{% for metric in metrics %}

    SELECT 'uniqueness' AS dq_dimension
         , '{{ metric.name }}' AS entity
         , ROUND(COUNT(DISTINCT {{ metric.distinct_expr }}) / t.total_count, 3) AS metric_value
         , CONCAT
                 (
                   'Unique count: '
                 , COUNT(DISTINCT {{ metric.distinct_expr }})
                 , ' Total count: '
                 , t.total_count
                 , ' Example value: '
                 , CASE WHEN COUNT(DISTINCT {{ metric.distinct_expr }}) < COUNT(*) 
                        THEN 
                             (
                                SELECT {{ metric.example_expr }}
                                  FROM products
                              GROUP BY {% for field in metric.group_by %}{{ field }}{% if not loop.last %}, {% endif %}{% endfor %}
                                HAVING COUNT(*) > 1
                              ORDER BY COUNT(*) DESC
                                 LIMIT 1
                             )
                         END
                 ) AS details                         
      FROM products
CROSS JOIN
          (
            SELECT COUNT(*) AS total_count 
              FROM products
          )t
  GROUP BY t.total_count
{% if not loop.last %}
 UNION ALL
{% endif %}
{% endfor %}
  ORDER BY metric_value DESC;
"""
template = Template(sql_template)
uniqueness_sql = template.render(metrics=metrics_config)

uniqueness = con.execute(uniqueness_sql).fetchdf()
uniqueness

,dq_dimension,entity,metric_value,details
0,uniqueness,product_id,1.000,Unique count: 10300 Total count: 10300 Example value:
1,uniqueness,category + price + brand,0.984,Unique count: 10133 Total count: 10300 Example value: Электроника>Умные часы | 0 | NULL (3 times)
2,uniqueness,product_name,0.251,Unique count: 2588 Total count: 10300 Example value: Samsung Клавиатуры Lite (25 раз)
3,uniqueness,product_name + brand,0.251,Unique count: 2588 Total count: 10300 Example value: Samsung Клавиатуры Lite | Samsung (25 times)


In [1464]:
# ============================================
# 3. ВАЛИДНОСТЬ (VALIDITY)
# ============================================

# Правила валидации для числовых полей: 
#Цена должна быть > 0
#Скидка должна быть 0-100%
#Рейтинг должен быть 1-5
#Отзывов не может быть < 0
#Итоговая цена должна быть >= 0

# Конфигурация правил
rules = [
    ('price', 'Цена должна быть > 0', 'price <= 0'),
    ('discount_pct', 'Скидка должна быть 0-100%', 'discount_pct NOT BETWEEN 0 AND 100'),
    ('rating', 'Рейтинг должен быть 1-5', 'rating NOT BETWEEN 1 AND 5'),
    ('reviews_count', 'Отзывов не может быть < 0', 'reviews_count < 0'),
    ('final_price', 'Итоговая цена должна быть >= 0', 'final_price < 0')
]

# шаблон
template = Template("""
{% for col, desc, cond in rules %}
    SELECT 'validity' as dq_dimension
         , '{{ col }} || {{ desc }}' as entity
         , 1.0 - COUNT(CASE WHEN {{ cond }} THEN 1 END) / COUNT(*) as metric_value
         , CONCAT
                 (
                   'Invalid count: ', COUNT(CASE WHEN {{ cond }} THEN 1 END)
                 , ' Total count: ', COUNT(*)
                 , ' Example value: [', 
                                       COALESCE
                                               (
                                                 (
                                                  SELECT STRING_AGG({{ col }}::TEXT, ', ') 
                                                    FROM 
                                                        (
                                                           SELECT {{ col }} 
                                                             FROM products 
                                                            WHERE {{ cond }} 
                                                         ORDER BY product_id 
                                                            LIMIT 3
                                                        ) t
                                                 )
                                              , 'None'
                                             ),
                                     ']'
                 ) as details
      FROM products
{% if not loop.last %}UNION ALL{% endif %}
{% endfor %}
  ORDER BY metric_value DESC;
""")

query = template.render(rules=rules)
result = con.execute(query).fetchdf()
result

,dq_dimension,entity,metric_value,details
0,validity,reviews_count || Отзывов не может быть < 0,1.000000,Invalid count: 0 Total count: 10300 Example value: [None]
1,validity,price || Цена должна быть > 0,0.979709,"Invalid count: 209 Total count: 10300 Example value: [0, 0, 0]"
2,validity,final_price || Итоговая цена должна быть >= 0,0.969903,"Invalid count: 310 Total count: 10300 Example value: [-1173230, -299411, -1079105]"
3,validity,rating || Рейтинг должен быть 1-5,0.969806,"Invalid count: 311 Total count: 10300 Example value: [0.5, 6.0, 0.0]"
4,validity,discount_pct || Скидка должна быть 0-100%,0.948641,"Invalid count: 529 Total count: 10300 Example value: [999, -10, -5]"


In [1466]:
# ============================================
# 4. КОНСИСТЕНТНОСТЬ (CONSISTENCY)
# ============================================
#4 1 Проверка консистентности расчетов: 
# Конфигурация проверок консистентности - 2 бизнес-правила
consistency_checks = [
    {
        'entity': 'final_price = price * (1 - discount_pct/100)',
        'valid_condition': 'ABS(ROUND(final_price, 2) - ROUND(price * (1 - discount_pct/100), 2)) <= 1',
        'invalid_condition': 'ABS(ROUND(final_price, 2) - ROUND(price * (1 - discount_pct/100), 2)) > 1',
        'sample_expr': "CONCAT('product_id: ', product_id, ' final_price: ', final_price, ' price: ', price, ' discount_pct: ', discount_pct, ' expected discount price: ', ROUND(price * (1 - discount_pct/100), 2))"
    },
    # Добавьте другие проверки здесь, например:
     {
         'entity': 'final_price <= price',
         'valid_condition': 'final_price <= price',
         'invalid_condition': 'final_price > price',
         'sample_expr': "CONCAT('product_id: ', product_id, ' final_price: ', final_price, ' price: ', price, ' discount_pct: ', discount_pct, ' expected discount price: ', ROUND(price * (1 - discount_pct/100), 2))"
     }
]

# Шаблон для проверок консистентности
consistency_template = """
{% for check in checks %}

    SELECT 'consistency' as dq_dimension
         , '{{ check.entity }}' as entity
         , COUNT(CASE WHEN {{ check.valid_condition }} THEN 1 END) * 1.0 / COUNT(*) as metric_value
         , CONCAT
                 (
                   'Invalid count: '
                 , COUNT(CASE WHEN {{ check.invalid_condition }} THEN 1 END)
                 , ' Total count: '
                 , COUNT(*)
                 , ' Sample data: '
                 , COALESCE(MAX(CASE WHEN {{ check.invalid_condition }} THEN {{ check.sample_expr }} END), '')
                 ) as details
      FROM products
{% if not loop.last %}UNION ALL{% endif %}
{% endfor %}
  ORDER BY metric_value DESC;
"""

# Рендеринг и выполнение запроса
template = Template(consistency_template)
query = template.render(checks=consistency_checks)
consistency1 = con.execute(query).fetchdf()
consistency1

,dq_dimension,entity,metric_value,details
0,consistency,final_price <= price,0.978155,Invalid count: 225 Total count: 10300 Sample data: product_id: PRD200249 final_price: 0 price: -1 discount_pct: 45 expected discount price: -0.55
1,consistency,final_price = price * (1 - discount_pct/100),0.888544,Invalid count: 1148 Total count: 10300 Sample data: product_id: PRD200296 final_price: 26203 price: 32408 discount_pct: 19 expected discount price: 26250.48


In [1468]:
#4 2 Проверка консистентности брендов - может быть apple, Apple, APPLE и так далее, а мы ожидаем унифицированного названия
query = """
WITH precalc AS
(
    SELECT UPPER(TRIM(brand)) AS normalized_brand
         , ARRAY_AGG(DISTINCT brand) AS variations
         , CASE WHEN COUNT(DISTINCT brand) = 1 THEN 1 ELSE 0 END AS metric_value
      FROM products
     WHERE brand IS NOT NULL
  GROUP BY UPPER(TRIM(brand))
)
    SELECT 'consistency' AS dq_dimension
         , 'brand' AS entity
         , metric_value
         , CASE WHEN metric_value=0
                THEN CONCAT
                           (
                            'Normalized brand: '
                           , normalized_brand
                           , '; Variations: '
                           , CAST(variations AS VARCHAR)
                           ) 
                 END details
      FROM precalc
"""
consistency2 = con.execute(query).fetchdf()
consistency2

,dq_dimension,entity,metric_value,details
0,consistency,brand,1,None
1,consistency,brand,1,None
2,consistency,brand,1,None
3,consistency,brand,1,None
4,consistency,brand,0,"Normalized brand: APPLE; Variations: [APPLE, Apple, apple]"
5,consistency,brand,1,None
6,consistency,brand,1,None
7,consistency,brand,1,None
8,consistency,brand,0,"Normalized brand: SAMSUNG; Variations: [samsung, Samsung, SAMSUNG]"
9,consistency,brand,1,None


In [1470]:
#4 3 Предположим, что наш каталог формируется вручную на основании какого-то другого, какой-то мастердаты от брендов.
# Создадим каталог мастер-данных, изменим цены, добавим в него какие-то сущности, отсутствующие в проверяемом отчете

samsung_masterdata = df[df['brand']=='Samsung'][['product_id', 'final_price']].copy()
# меняем 5 случайных цен на ±15%
samsung_masterdata.loc[samsung_masterdata.sample(n=5).index, 'final_price'] *= np.random.uniform(0.85, 1.15, 5)
samsung_masterdata['final_price'] = samsung_masterdata['final_price'].round(2)
#добавляем какой-то продукт, которого нет в products
samsung_masterdata = pd.concat([samsung_masterdata, pd.Series({'product_id': 10011123, 'final_price': 10})], ignore_index=True)
con.register('samsung_masterdata', samsung_masterdata)

########################   Моделируем изменение нашего экземпляра каталога   ########################

samsung_products = """
     SELECT product_id
          , final_price
       FROM products 
      WHERE brand = 'Samsung'
  UNION ALL
     SELECT 12315232
          , 100 ----добавляем в products сущность, которой нет в мастер-данных
"""
samsung_products = Template(samsung_products)
samsung_products = samsung_products.render(columns=columns)
samsung_products = con.execute(samsung_products).fetchdf()
con.register('samsung_products', samsung_products)

########################   Сверка   ########################

query = """
WITH details AS 
(
     SELECT COALESCE(m.product_id, p.product_id) AS product_id
          , m.final_price AS masterdata_price
          , p.final_price AS catalog_price
          , ABS(COALESCE(m.final_price, 0) - COALESCE(p.final_price, 0)) AS discrepancy
          , CASE WHEN ABS(COALESCE(m.final_price, 0) - COALESCE(p.final_price, 0)) > 1 
                 THEN 0.00
                 ELSE 1.00
             END as is_valid
       FROM samsung_masterdata m
  FULL JOIN samsung_products p ON m.product_id = p.product_id
),

discrepancy_ranking AS 
(             
     SELECT FIRST_VALUE(
                CASE WHEN masterdata_price IS NULL AND catalog_price IS NOT NULL 
                     THEN CONCAT('Product ID: ', product_id, ' Masterdata price: Not found! Catalog price: ', catalog_price, ' Discrepancy: ', discrepancy, ' | ')
                END
           ) OVER (PARTITION BY 
                CASE WHEN masterdata_price IS NULL AND catalog_price IS NOT NULL THEN 1 END 
                ORDER BY discrepancy DESC
           ) AS top_missing_masterdata
          , FIRST_VALUE(
                CASE WHEN masterdata_price IS NOT NULL AND catalog_price IS NULL 
                     THEN CONCAT('Product ID: ', product_id, ' Masterdata price: ', masterdata_price, ' Catalog price: Not found! Discrepancy: ', discrepancy, ' | ')
                END
           ) OVER (PARTITION BY 
                CASE WHEN masterdata_price IS NOT NULL AND catalog_price IS NULL THEN 2 END 
                ORDER BY discrepancy DESC
           ) AS top_missing_catalog
          , FIRST_VALUE(
                CASE WHEN masterdata_price IS NOT NULL AND catalog_price IS NOT NULL AND ABS(masterdata_price - catalog_price) > 1 
                     THEN CONCAT('Product ID: ', product_id, ' Masterdata price: ', masterdata_price, ' Catalog price: ', catalog_price, ' Discrepancy: ', discrepancy, ' | ')
                END
           ) OVER (PARTITION BY 
                CASE WHEN masterdata_price IS NOT NULL AND catalog_price IS NOT NULL AND ABS(masterdata_price - catalog_price) > 1 THEN 3 END 
                ORDER BY discrepancy DESC
           ) AS top_price_discrepancy
        FROM details
       WHERE masterdata_price IS NULL 
          OR catalog_price IS NULL 
          OR ABS(masterdata_price - catalog_price) > 1
)

     SELECT 'consistency' AS dq_dimension
          , 'Samsung/final_price' AS entity
          , t.metric_value
          , CONCAT
                  (
                   'Discrepancies found: '
                  , t.discrepancy_cnt
                  , ' || Total count: '
                  , t.total_cnt
                  , ' || Example: '
                  , COALESCE(MAX(r.top_missing_masterdata), '')
                  , COALESCE(MAX(r.top_missing_catalog), '')
                  , COALESCE(MAX(r.top_price_discrepancy), '')
                  ) AS details
      FROM discrepancy_ranking r
CROSS JOIN 
          (
           SELECT AVG(is_valid) AS metric_value
                , COUNT(*) AS total_cnt
                , COUNT(CASE WHEN masterdata_price IS NULL OR catalog_price IS NULL OR ABS(masterdata_price - catalog_price) > 1 THEN 1 END) AS discrepancy_cnt
             FROM details
          ) t
  GROUP BY t.metric_value
         , t.total_cnt
         , t.discrepancy_cnt
"""
consistency3 = con.execute(query).fetchdf()
consistency3

,dq_dimension,entity,metric_value,details
0,consistency,Samsung/final_price,0.994545,Discrepancies found: 8 || Total count: 1100 || Example: Product ID: 12315232 Masterdata price: Not found! Catalog price: 100 Discrepancy: 100.0 | Product ID: PRD106776 Masterdata price: 131541.35 Catalog price: 121995 Discrepancy: 9546.350000000006 |


In [1472]:
# ============================================
# 5. ТОЧНОСТЬ И ВЫБРОСЫ (ACCURACY)
# ============================================
#правило 3 сигм
numeric_cols = ['price', 'discount_pct', 'final_price', 'rating', 'reviews_count']

query_template="""
WITH precalc AS
(
    SELECT col entity
         , ROUND(AVG(val), 2) avg_
         , ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY val), 2) as median_
         , ROUND(STDDEV(val), 2) as stdev_
         , ROUND(MIN(val), 2) as min_
         , ROUND(MAX(val), 2) as max_
         , ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY val), 2) q25
         , ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY val), 2) q75
      FROM 
          (
            {%- for col in columns -%}
            SELECT '{{ col }}' as col, {{ col }} as val 
              FROM products
            {%- if not loop.last %} UNION ALL {% endif -%}
            {%- endfor -%}
          ) t
     WHERE val IS NOT NULL
  GROUP BY col
)
    SELECT 'accuracy' AS dq_dimension
         , entity
         , CASE WHEN stdev_ = 0 
                THEN 1
                ELSE
                    CASE WHEN ABS((avg_ - median_) / stdev_) <= 3 
                         THEN 1
                         ELSE 0
                          END
                 END AS metric_value
         , CONCAT
                 (
                   'Average: ', avg_
                 , ' Median: ', median_
                 , ' StdDev: ', stdev_
                 , ' Min: ', min_
                 , ' Max: ', max_
                 , ' Q25: ', q25
                 , ' Q75: ', q75
                 ) AS details
      FROM precalc
  """
template = Template(query_template)
query = template.render(columns=numeric_cols)
accuracy = con.execute(query).fetchdf()
accuracy

,dq_dimension,entity,metric_value,details
0,accuracy,rating,1,Average: 3.78 Median: 3.8 StdDev: 0.98 Min: 0.0 Max: 10.0 Q25: 3.1 Q75: 4.4
1,accuracy,discount_pct,1,Average: 46.06 Median: 35.0 StdDev: 97.85 Min: -10.0 Max: 999.0 Q25: 16.0 Q75: 54.0
2,accuracy,final_price,1,Average: 52642.26 Median: 55391.5 StdDev: 114604.41 Min: -1778599.0 Max: 219855.0 Q25: 24577.5 Q75: 93034.25
3,accuracy,reviews_count,1,Average: 253.12 Median: 254.0 StdDev: 143.86 Min: 0.0 Max: 500.0 Q25: 130.0 Q75: 378.0
4,accuracy,price,1,Average: 97938.37 Median: 98085.0 StdDev: 58768.62 Min: -1.0 Max: 199998.0 Q25: 46582.75 Q75: 148418.75


In [1474]:
# ============================================
# 6. ЦЕЛОСТНОСТЬ (INTEGRITY)
# ============================================
integrity = Template("""
{% set max_examples = max_examples|default(3) %}
{% set example_truncate_len = example_truncate_len|default(30) %}

{% set rules = [
    {
        'id': 1,
        'name': 'Товары не в наличии с ценой 0',
        'condition': 'p.is_in_stock = 0 AND p.price = 0'
    },
    {
        'id': 2,
        'name': 'Скидка ведет к увеличению цены',
        'condition': 'p.discount_pct > 0 AND p.final_price > p.price'
    },
    {
        'id': 3,
        'name': 'Скидка 0%, но final_price ≠ price',
        'condition': 'p.discount_pct = 0 AND p.final_price != p.price'
    },
    {
        'id': 4,
        'name': 'Товары с отзывами, но без рейтинга',
        'condition': 'p.reviews_count > 0 AND p.rating IS NULL'
    }
] %}

WITH validation_rules AS 
(
    SELECT rule_id
         , rule_name
      FROM 
          (
           VALUES
           {% for rule in rules %}
           ({{ rule.id }}, '{{ rule.name }}'){% if not loop.last %},{% endif %}
           {% endfor %}
          ) AS rules(rule_id, rule_name)
),

rule_violations AS 
(
    {% for rule in rules %}
    SELECT {{ rule.id }} AS rule_id
         , p.product_id
         , p.product_name
         , ROW_NUMBER() OVER (PARTITION BY {{ rule.id }} ORDER BY p.product_id) as example_rn
      FROM products p
     WHERE {{ rule.condition }}
    {% if not loop.last %}UNION ALL{% endif %}
    {% endfor %}
),

rule_summary AS (
    SELECT rv.rule_id
         , COUNT(rv.product_id) as violation_count
         , STRING_AGG(
               CASE WHEN rv.example_rn <= {{ max_examples }} 
                    THEN LEFT(rv.product_name, {{ example_truncate_len }})
                    END, 
               '; ' ORDER BY rv.product_id
           ) as examples
      FROM rule_violations rv
  GROUP BY rv.rule_id
),

total_products AS (
    SELECT COUNT(*) as total_count 
      FROM products
)

    SELECT 'integrity'  AS dq_dimension
         , vr.rule_name as entity
         , 1.0 - (COALESCE(rs.violation_count, 0)::FLOAT / tp.total_count) as metric_value
         , CONCAT
                (
                 'Invalid count: ', COALESCE(rs.violation_count, 0), 
                 ' Total count: ', tp.total_count
                 ,
         '. Sample value: ',
         CASE 
             WHEN COALESCE(rs.violation_count, 0) = 0 THEN 'Нет нарушений'
             ELSE COALESCE(rs.examples, '') || 
                  CASE WHEN rs.violation_count > {{ max_examples }} 
                       THEN '... (' || (rs.violation_count - {{ max_examples }}) || ' more)' 
                       ELSE '' 
                  END
         END
       ) as details
      FROM validation_rules vr
 LEFT JOIN rule_summary rs ON vr.rule_id = rs.rule_id
CROSS JOIN total_products tp
  ORDER BY vr.rule_id;
""")
query = integrity.render(columns=numeric_cols)
integrity = con.execute(query).fetchdf()
integrity

,dq_dimension,entity,metric_value,details
0,integrity,Товары не в наличии с ценой 0,0.979806,Invalid count: 208 Total count: 10300. Sample value: LG Умные часы Lite; Asus Видеокарты Lite; Philips Умные часы Lite... (205 more)
1,integrity,Скидка ведет к увеличению цены,0.999515,Invalid count: 5 Total count: 10300. Sample value: Ультра Стиральные машины Pro; Apple Видеокарты Edition; APPLE Процессоры Lite... (2 more)
2,integrity,"Скидка 0%, но final_price ≠ price",0.998058,Invalid count: 20 Total count: 10300. Sample value: Huawei Холодильники Max; Apple Ноутбуки Lite; Xiaomi Планшеты Edition... (17 more)
3,integrity,"Товары с отзывами, но без рейтинга",1.000000,Invalid count: 0 Total count: 10300. Sample value: Нет нарушений


In [1476]:
# ============================================
# 7. СООТВЕТСТВИЕ СТАНДАРТАМ (CONFORMITY)
# ============================================
## 7.1. ПРОВЕРКА НАЗВАНИЙ НА СПАМ
query="""
WITH total_products AS 
(
     SELECT COUNT(*) AS total_count 
       FROM products
),

spam_patterns AS 
(
    SELECT pattern
         , description
      FROM 
          (
           VALUES
                ('[ХИТ', 'Маркер "ХИТ" в названии'),
                ('🔥', 'Эмодзи огонек'),
                ('ЛУЧШАЯ ЦЕНА', 'Капслок маркеры'),
                ('РАСПРОДАЖА', 'Маркеры распродажи'),
                ('СКИДКА', 'Упоминание скидки в названии'),
                ('🎁', 'Эмодзи подарка'),
                ('!!!', 'Множественные восклицательные знаки')
          ) AS patterns(pattern, description)
),
product_matches AS 
(
    SELECT sp.pattern
         , sp.description
         , p.product_id
         , p.product_name
         , ROW_NUMBER() OVER (PARTITION BY sp.pattern ORDER BY p.product_id) as rn  ----для получения примера некачественных данных
      FROM spam_patterns sp
 LEFT JOIN products p ON UPPER(p.product_name) LIKE '%' || UPPER(sp.pattern) || '%'
)
    SELECT 'conformity' AS dq_dimension
         , pm.description AS entity
         , 1.0 - COUNT(pm.product_id)::FLOAT / tp.total_count AS metric_value
         , CONCAT
                 (
                   'Invalid count: ', COUNT(pm.product_id)
                 , ' Total count: ', tp.total_count
                 , ' Example: '
                 , COALESCE(MAX(CASE WHEN pm.rn = 1 THEN LEFT(pm.product_name, 200) || '...' END), '')
                 ) AS details
      FROM product_matches pm
CROSS JOIN total_products tp
  GROUP BY pm.description
         , pm.pattern
         , tp.total_count
  ORDER BY metric_value ASC;
"""
conformity1 = con.execute(query).fetchdf()
conformity1

,dq_dimension,entity,metric_value,details
0,conformity,"Маркер ""ХИТ"" в названии",0.950000,Invalid count: 515 Total count: 10300 Example: ЛУЧШАЯ ЦЕНА!!! Lenovo Микроволновки Edition [ХИТ ПРОДАЖ]...
1,conformity,Эмодзи подарка,0.950485,Invalid count: 510 Total count: 10300 Example: 🔥ГОРЯЧЕЕ ПРЕДЛОЖЕНИЕ🔥 samsung Видеокарты Edition 🎁ПОДАРОК🎁...
2,conformity,Маркеры распродажи,0.950777,Invalid count: 507 Total count: 10300 Example: 🎁ПОДАРОК🎁 Apple Смартфоны Lite РАСПРОДАЖА...
3,conformity,Упоминание скидки в названии,0.953883,Invalid count: 475 Total count: 10300 Example: РАСПРОДАЖА Samsung Процессоры Edition СКИДКА 90%...
4,conformity,Капслок маркеры,0.955825,Invalid count: 455 Total count: 10300 Example: ЛУЧШАЯ ЦЕНА!!! Lenovo Микроволновки Edition [ХИТ ПРОДАЖ]...
5,conformity,Множественные восклицательные знаки,0.955825,Invalid count: 455 Total count: 10300 Example: ЛУЧШАЯ ЦЕНА!!! Lenovo Микроволновки Edition [ХИТ ПРОДАЖ]...
6,conformity,Эмодзи огонек,0.956408,Invalid count: 449 Total count: 10300 Example: 🔥ГОРЯЧЕЕ ПРЕДЛОЖЕНИЕ🔥 samsung Видеокарты Edition 🎁ПОДАРОК🎁...


In [1478]:
##7.2. ПРОВЕРКА ДЛИНЫ ТЕКСТОВЫХ ПОЛЕЙ

text_fields_config = [
    {'entity': 'product_name', 'column': 'product_name', 'too_short': 5, 'too_long': 25},
    {'entity': 'description', 'column': 'description', 'too_short': 10, 'too_long': 100},
    {'entity': 'brand', 'column': 'brand', 'too_short': 2, 'too_long': 10},
    {'entity': 'category_path', 'column': 'category_path', 'too_short': 3, 'too_long': 100}
]

# Используем шаблон Jinja2 для параметризации
sql_template = """
{% set quality_threshold = 0.95 %}

WITH text_stats AS 
(
    {% for field in text_fields %}
    SELECT '{{ field.entity }}' as entity
         , ROUND(AVG(LENGTH(COALESCE({{ field.column }}, ''))), 1) as avg_len
         , MIN(LENGTH(COALESCE({{ field.column }}, ''))) as min_len
         , MAX(LENGTH(COALESCE({{ field.column }}, ''))) as max_len
         , COUNT(CASE WHEN LENGTH(COALESCE({{ field.column }}, '')) < {{ field.too_short }} THEN 1 END) as cnt_too_short
         , COUNT(CASE WHEN LENGTH(COALESCE({{ field.column }}, '')) > {{ field.too_long }} THEN 1 END) as cnt_too_long
         , COUNT(*) cnt
      FROM products    
    {% if not loop.last %}UNION ALL{% endif %}
    {% endfor %}
)
    SELECT 'conformity' as dq_dimension
         , entity
         , CASE WHEN 1.0 - (cnt_too_short::FLOAT / cnt) < {{ quality_threshold }}
                  OR 1.0 - (cnt_too_long::FLOAT / cnt) < {{ quality_threshold }}
                THEN 0
                ELSE 1
                  END as metric_value
         , CONCAT('Средняя длина: ', avg_len, 
         ' Минимальная длина: ', min_len, 
         ' Максимальная длина: ', max_len, 
         ' Количество слишком коротких значений: ', cnt_too_short, 
         ' Количество слишком длинных значений: ', cnt_too_long) as details
      FROM text_stats
  ORDER BY metric_value DESC
"""
template = Template(sql_template)
conformity2 = template.render(text_fields=text_fields_config)

# Выполняем запрос
conformity2 = con.execute(conformity2).fetchdf()
conformity2

,dq_dimension,entity,metric_value,details
0,conformity,category_path,1,Средняя длина: 28.9 Минимальная длина: 20 Максимальная длина: 50 Количество слишком коротких значений: 0 Количество слишком длинных значений: 0
1,conformity,brand,0,Средняя длина: 5.0 Минимальная длина: 0 Максимальная длина: 7 Количество слишком коротких значений: 835 Количество слишком длинных значений: 0
2,conformity,product_name,0,Средняя длина: 26.2 Минимальная длина: 15 Максимальная длина: 76 Количество слишком коротких значений: 0 Количество слишком длинных значений: 2815
3,conformity,description,0,Средняя длина: 88.5 Минимальная длина: 0 Максимальная длина: 279 Количество слишком коротких значений: 1039 Количество слишком длинных значений: 3951


In [1480]:
# Ваш список датафреймов
dq_checks = [
    completeness,
    uniqueness,
    validity,
    consistency1,
    consistency2,
    consistency3,
    accuracy,
    integrity,
    conformity1,
    conformity2
]

combined_dq_checks = pd.concat(dq_checks, ignore_index=True)
combined_dq_checks

,dq_dimension,entity,metric_value,details
0,completeness,specifications,0.880000,Non-nulls: 9059 Total rows: 10300
1,completeness,description,0.899000,Non-nulls: 9261 Total rows: 10300
2,completeness,brand,0.919000,Non-nulls: 9465 Total rows: 10300
3,completeness,product_id,1.000000,Non-nulls: 10300 Total rows: 10300
4,completeness,product_name,1.000000,Non-nulls: 10300 Total rows: 10300
5,completeness,category_path,1.000000,Non-nulls: 10300 Total rows: 10300
6,completeness,seller_id,1.000000,Non-nulls: 10300 Total rows: 10300
7,completeness,price,1.000000,Non-nulls: 10300 Total rows: 10300
8,completeness,discount_pct,1.000000,Non-nulls: 10300 Total rows: 10300
9,completeness,final_price,1.000000,Non-nulls: 10300 Total rows: 10300
